In [ ]:
!pip -q install transformers datasets accelerate

In [ ]:
import torch
from datasets import Dataset
from transformers import (AutoTokenizer, AutoModelForCausalLM, DataCollatorForLanguageModeling,
                          TrainingArguments, Trainer, set_seed)
set_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

In [ ]:
# Custom dataset (replace with your own text: lines = open("mytext.txt").read().split("\n"))
lines = [
    "Machine learning helps students analyse data and build smart applications.",
    "In our college lab, students train neural networks on small datasets every week.",
    "Natural language processing allows computers to read, understand and write human language.",
    "A good dataset is more important than a complex model.",
    "Students present their projects at the annual technical festival of the college.",
    "Python is the favourite programming language of data science students.",
    "The professor explained how transformers use attention to understand sentences.",
    "Before training a model we clean the data and remove missing values.",
    "Deep learning models need a lot of data and powerful GPUs.",
    "The library stays open late during exam weeks so students can study together.",
    "A chatbot can answer simple questions of students about courses and schedules.",
    "Teamwork and regular practice make every student a better programmer.",
    "Cloud platforms like Google Colab give free GPUs to learners.",
    "Language models predict the next word by learning patterns from large text collections.",
    "Every semester the department organises workshops on artificial intelligence.",
] * 4
dataset = Dataset.from_dict({"text": lines})
print(len(dataset), "training lines")

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained("gpt2").to(device)

def generate(prompt, n=2, max_new_tokens=40):
    model.eval()
    enc = tokenizer(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        out = model.generate(**enc, do_sample=True, top_k=50, top_p=0.95, temperature=0.8,
                             max_new_tokens=max_new_tokens, num_return_sequences=n,
                             pad_token_id=tokenizer.eos_token_id)
    return [tokenizer.decode(o, skip_special_tokens=True) for o in out]

prompt = "In our college lab, students"
print("===== BEFORE fine-tuning =====")
for t in generate(prompt): print("-", t.replace("\n", " "))

In [ ]:
def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, max_length=64)

tokenized = dataset.map(tokenize, batched=True, remove_columns=["text"])
collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

args = TrainingArguments(
    output_dir="gpt2_custom",
    num_train_epochs=6,
    per_device_train_batch_size=4,
    learning_rate=5e-5,
    logging_steps=20,
    save_strategy="no",
    fp16=torch.cuda.is_available(),
    report_to="none",
)
trainer = Trainer(model=model, args=args, train_dataset=tokenized, data_collator=collator)
trainer.train()

In [ ]:
print("===== AFTER fine-tuning =====")
for p in ["In our college lab, students", "Natural language processing", "A chatbot can"]:
    print(f"\nPrompt: {p}")
    for t in generate(p): print("-", t.replace("\n", " "))

In [ ]:
# Effect of decoding settings
p = "Machine learning helps"
enc = tokenizer(p, return_tensors="pt").to(device)
model.eval()
with torch.no_grad():
    greedy = model.generate(**enc, max_new_tokens=30, do_sample=False, pad_token_id=tokenizer.eos_token_id)
    beam = model.generate(**enc, max_new_tokens=30, num_beams=5, no_repeat_ngram_size=2, early_stopping=True,
                          pad_token_id=tokenizer.eos_token_id)
print("Greedy:", tokenizer.decode(greedy[0], skip_special_tokens=True))
print("Beam  :", tokenizer.decode(beam[0], skip_special_tokens=True))
model.save_pretrained("gpt2_custom_model"); tokenizer.save_pretrained("gpt2_custom_model")